# 01 · Data Preparation
### Lifestyle Archetypes and Problematic Internet Use — Pipeline Notebook 1 of 5

**Question:** Given patterns of sleep, physical activity and body composition, do distinct lifestyle profiles emerge among children and adolescents, and how do these profiles relate to problematic internet use (PIU) and mental health screening scores?

**This notebook covers:**
1. Data dictionary review
2. Missingness documentation
3. Actigraphy (accelerometer) feature extraction
4. Feature engineering & merge
5. Exploratory data analysis (pre-standardisation)

**Pipeline position:** first of five notebooks. Produces `01_data_preparation.pkl`, consumed by `02_dimensionality_reduction_clustering.ipynb`.

| Notebook | Purpose |
|---|---|
| **01_data_preparation** | *(this notebook)* raw data → clean modelling table |
| 02_dimensionality_reduction_clustering | UMAP embeddings + K-means / Agglomerative / DBSCAN |
| 03_validation_stability | internal validation metrics + stability checks, selects final method |
| 04_archetype_characterisation | describes archetypes, relates them to PIU / mental health, visual summary |
| 05_three_group_alternative_analysis | **independent, exploratory** check of a forced 3-group solution — flagged as needing independent confirmation |

**Responsible-analysis boundary (carried through every notebook in this pipeline):**
- This is an **unsupervised, exploratory segmentation** task, not a predictive or diagnostic modelling task.
- PIU and mental health scores are **screening instrument results, not clinical diagnoses**.
- Clusters describe **population-level lifestyle patterns**, never individual-level labels or diagnoses.
- Any outcome association is **preliminary and subset-derived** (train-set labels only), reported as association, not validated clinical finding.


## Setup & Environment

In [ ]:
import os
import glob
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer

import pyarrow.parquet as pq


In [ ]:
!pip -q install pyarrow missingno


### Paths & artifact directory

All intermediate outputs from this pipeline are written to `ARTIFACT_DIR` as pickled dictionaries, one per notebook, so each downstream notebook can be run independently (given the artifacts exist) without re-running everything upstream.

In [ ]:
DATA_DIR = Path(
    "/kaggle/input/competitions/child-mind-institute-problematic-internet-use"
)

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
DATA_DICTIONARY = DATA_DIR / "data_dictionary.csv"
SAMPLE_SUBMISSION = DATA_DIR / "sample_submission.csv"

SERIES_DIR = DATA_DIR / "series_train.parquet"
SERIES_TEST_DIR = DATA_DIR / "series_test.parquet"

CACHE_PATH = Path("/kaggle/working/actigraphy_summary.csv")

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
dictionary = pd.read_csv(DATA_DICTIONARY)


## 1. Understanding the Dataset (Data Dictionary Review)

Before creating any new features or running any analysis, it is important to understand what each group of variables represents. The **Child Mind Institute (CMI) Healthy Brain Network** dataset organises variables into different sections based on the questionnaire, assessment, or measurement from which they were collected.

The main groups of variables are:

| Variable Group | Full Name | What It Contains |
|----------------|-----------|------------------|
| **Basic_Demos** | Basic Demographics | Participant information such as age, sex, and basic background characteristics. |
| **CGAS** | Children's Global Assessment Scale | A clinician-rated measure of a child's overall psychological, social, and school functioning. Higher scores generally indicate better functioning. |
| **Physical** | Physical Measurements | Height, weight, Body Mass Index (BMI), waist circumference, blood pressure, heart rate, and other physical health measurements. |
| **FGC** | FitnessGram Components | Measures of physical fitness such as grip strength, flexibility, muscular endurance, and cardiovascular fitness. |
| **BIA** | Bioelectrical Impedance Analysis | Body composition measurements including body fat percentage, lean muscle mass, total body water, and related metrics. |
| **PAQ_A** | Physical Activity Questionnaire for Adolescents | A self-reported questionnaire measuring the physical activity levels of adolescents (typically ages 14–20) over the previous seven days. |
| **PAQ_C** | Physical Activity Questionnaire for Children | A child-friendly version of the physical activity questionnaire designed for younger participants (typically ages 8–14). |
| **PCIAT** | Parent–Child Internet Addiction Test | A questionnaire assessing problematic internet use (PIU). It measures behaviours such as excessive internet use, difficulty controlling internet use, and its impact on daily life. This is the primary outcome used later to compare lifestyle archetypes—it is **not** used to create the clusters themselves. |
| **SDS** | Sleep Disturbance Scale for Children | A questionnaire measuring different aspects of children's sleep, including sleep quality, sleep disorders, and disturbances. |
| **PreInt_EduHx** | Prenatal, Intervention and Educational History | Information about prenatal history, developmental milestones, educational support, previous interventions, and learning history. |

### Accelerometer (Wearable) Data

In addition to the questionnaire and clinical measurements, the dataset includes wearable accelerometer data stored separately as:

- `series_train.parquet`
- `series_test.parquet`

These files contain continuous movement data collected from wearable devices (actigraphy). Unlike the questionnaire data, they record participants' activity throughout the day and night.

From these time-series data, we can engineer lifestyle features such as:

- Average nightly sleep duration
- Night-to-night sleep variability
- Physical activity intensity
- Daily activity consistency
- Circadian (day–night) activity patterns

These engineered features provide objective measurements of participants' lifestyles and form the basis of the clustering analysis used to identify different lifestyle archetypes.

In [ ]:
print("train shape:", train.shape)
print("test shape :", test.shape)
print("dictionary shape:", dictionary.shape)

display(dictionary.head(20))

# Column families present in train, grouped by instrument prefix
families = sorted(set(c.split("-")[0] for c in train.columns if "-" in c))
print("\nInstrument families in train.csv:")
for fam in families:
    cols = [c for c in train.columns if c.startswith(fam + "-")]
    print(f"  {fam:15s} -> {len(cols)} columns")


## 2. Missingness Documentation

Several instruments (FitnessGram, BIA, PAQ) were only administered to subsets of participants (e.g. `PAQ_A` for adolescents, `PAQ_C` for children), so missingness is expected to be structured rather than random. We document it explicitly before deciding how to handle it, per the responsible-analysis requirement to make every data decision transparent.

In [ ]:
missing = (
    train.isna().mean().sort_values(ascending=False).mul(100).round(1)
    .rename("pct_missing")
    .to_frame()
)
missing["n_missing"] = train.isna().sum().reindex(missing.index)
display(missing.head(30))

plt.figure(figsize=(8, 10))
top_missing = missing.head(30)
sns.barplot(x=top_missing["pct_missing"], y=top_missing.index, color="#4C72B0")
plt.xlabel("% missing")
plt.title("Top 30 columns by missingness (train.csv)")
plt.tight_layout()
plt.show()

try:
    import missingno as msno
    msno.matrix(train.sample(min(300, len(train)), random_state=42))
    plt.title("Missingness pattern (random sample of 300 rows)")
    plt.show()
except ImportError:
    print("missingno not available - skipping matrix plot")


**Observation to carry forward:** questionnaire-instrument missingness (PAQ, FGC, BIA) is largely structural (age-gated instruments, or a participant simply not completing a station), while the accelerometer series is present for only a subset of participants (`series_train.parquet` has one folder per child who wore the device; not every `id` in `train.csv` has a matching recording). We handle this explicitly below: children without actigraphy retain missing wearable features and are imputed at the model-input stage, with imputation applied only to clustering *inputs*, never to the outcome variables used later for characterisation.

## 3. Actigraphy (Accelerometer) Feature Extraction

`series_train.parquet` stores one sub-folder per child (`id=<participant_id>/part-0.parquet`) with 5-second epoch-level accelerometer data: `X`, `Y`, `Z`, `enmo` (Euclidean Norm Minus One, a standard activity-intensity metric), `anglez`, `light`, `battery_voltage`, `non-wear_flag`, `time_of_day` (nanoseconds since midnight), `weekday` (1=Monday...7=Sunday) and `relative_date_PCIAT` (day index relative to the PCIAT assessment).

We reduce each child's raw time series to a small number of **lifestyle-relevant summary features**: how much they sleep, how variable that sleep is, how physically active they are, how variable that activity is, and how different their weekday vs weekend activity looks. This mirrors the "reduce dimensionality" step suggested for the wearable data before it ever reaches UMAP.

**Method transparency (documented simplifications):**
- Epochs are 5 seconds long (`epoch_seconds = 5`).
- Non-wear epochs (`non-wear_flag == 1`) are excluded from all calculations.
- A day is only used if it has at least 16 hours of valid wear time (`min_valid_hours_per_day = 16`), to avoid days dominated by device removal biasing sleep/activity estimates.
- "Night-time" is defined as clock hours 22:00-08:00, derived from `time_of_day`. This is a fixed-window heuristic, **not** a validated polysomnography-grade sleep-staging algorithm.
- An epoch is flagged "low activity" if `enmo < 0.02` (g), a threshold consistent with common wrist-worn accelerometer sleep-detection conventions in the actigraphy literature. Sleep duration for a night is the count of low-activity epochs within the night window, converted to hours.
- "Weekday" = days 1-5, "weekend" = days 6-7 (`weekday` column).

This is intentionally a pragmatic, well-documented simplification appropriate for population-level segmentation - not a clinical sleep score.

In [ ]:
EPOCH_SECONDS = 5
LOW_ACTIVITY_ENMO_THRESHOLD = 0.02
MIN_VALID_HOURS_PER_DAY = 16
NIGHT_START_HOUR = 22
NIGHT_END_HOUR = 8


def _hour_of_day(time_of_day_ns):
    """Convert the 'time_of_day' nanoseconds-since-midnight column to a
    fractional clock hour in [0, 24)."""
    return (time_of_day_ns / 1e9) / 3600.0


def extract_actigraphy_features(participant_dir):
    """Summarise one child's accelerometer recording into lifestyle features.

    Returns a dict of features, or None if the recording has no usable
    (sufficiently-worn) days.
    """
    parquet_files = glob.glob(os.path.join(str(participant_dir), "*.parquet"))
    if not parquet_files:
        return None

    df = pq.read_table(parquet_files[0]).to_pandas()
    if "non-wear_flag" in df.columns:
        df = df[df["non-wear_flag"] == 0]
    if df.empty:
        return None

    df["hour"] = _hour_of_day(df["time_of_day"].to_numpy())
    df["is_night"] = (df["hour"] >= NIGHT_START_HOUR) | (df["hour"] < NIGHT_END_HOUR)
    df["is_weekend"] = df["weekday"].isin([6, 7])

    # keep only days with sufficient valid wear time
    epochs_per_day = df.groupby("relative_date_PCIAT").size()
    valid_hours_per_day = epochs_per_day * EPOCH_SECONDS / 3600.0
    valid_days = valid_hours_per_day[valid_hours_per_day >= MIN_VALID_HOURS_PER_DAY].index
    df = df[df["relative_date_PCIAT"].isin(valid_days)]
    if df.empty:
        return None

    # --- sleep duration per night ---
    night_df = df[df["is_night"]]
    sleep_epochs = night_df[night_df["enmo"] < LOW_ACTIVITY_ENMO_THRESHOLD]
    sleep_hours_per_day = (
        sleep_epochs.groupby("relative_date_PCIAT").size() * EPOCH_SECONDS / 3600.0
    )
    sleep_duration_mean = sleep_hours_per_day.mean() if len(sleep_hours_per_day) else np.nan
    sleep_duration_std = sleep_hours_per_day.std() if len(sleep_hours_per_day) > 1 else np.nan

    # --- daytime activity intensity ---
    day_df = df[~df["is_night"]]
    daily_mean_enmo = day_df.groupby("relative_date_PCIAT")["enmo"].mean()
    activity_intensity_mean = daily_mean_enmo.mean() if len(daily_mean_enmo) else np.nan
    activity_intensity_std = daily_mean_enmo.std() if len(daily_mean_enmo) > 1 else np.nan

    # --- weekday vs weekend consistency ---
    weekday_mean = day_df.loc[~day_df["is_weekend"], "enmo"].mean()
    weekend_mean = day_df.loc[day_df["is_weekend"], "enmo"].mean()
    if pd.notna(weekday_mean) and weekday_mean > 0 and pd.notna(weekend_mean):
        weekday_weekend_ratio = weekend_mean / weekday_mean
    else:
        weekday_weekend_ratio = np.nan

    n_valid_days = len(valid_days)

    return {
        "sleep_duration_mean": sleep_duration_mean,
        "sleep_duration_std": sleep_duration_std,
        "activity_intensity_mean": activity_intensity_mean,
        "activity_intensity_std": activity_intensity_std,
        "weekday_weekend_activity_ratio": weekday_weekend_ratio,
        "n_valid_actigraphy_days": n_valid_days,
    }


In [ ]:
if CACHE_PATH.exists():
    actigraphy_summary = pd.read_csv(CACHE_PATH)
    print(f"Loaded cached actigraphy summary: {actigraphy_summary.shape}")
else:
    participant_dirs = sorted(glob.glob(str(SERIES_DIR / "id=*")))
    print(f"Found {len(participant_dirs)} participants with actigraphy recordings")

    try:
        from tqdm.notebook import tqdm
    except ImportError:
        tqdm = lambda x, **kw: x

    records = []
    for pdir in tqdm(participant_dirs, desc="Extracting actigraphy features"):
        participant_id = os.path.basename(pdir).replace("id=", "")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            feats = extract_actigraphy_features(pdir)
        if feats is not None:
            feats["id"] = participant_id
            records.append(feats)

    actigraphy_summary = pd.DataFrame.from_records(records)
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    actigraphy_summary.to_csv(CACHE_PATH, index=False)
    print(f"Extracted and cached actigraphy summary: {actigraphy_summary.shape}")

display(actigraphy_summary.head())
print("\nCoverage: {}/{} train participants have usable actigraphy features".format(
    train["id"].isin(actigraphy_summary["id"]).sum(), len(train)
))


## 4. Feature Engineering & Merge

We now merge the actigraphy summary onto the tabular data and assemble the final **lifestyle feature set** used for clustering. Following the brief, demographics (age, sex) and outcome measures (PIU / mental health scores) are deliberately **excluded from the clustering inputs** - they are held back for the characterisation notebook (04) so that the segmentation reflects lifestyle only, not the outcomes we later relate it to.

**Clustering input features:**

| Feature | Source | Represents |
|---|---|---|
| `sleep_duration_mean`, `sleep_duration_std` | Actigraphy | Sleep amount & regularity |
| `activity_intensity_mean`, `activity_intensity_std` | Actigraphy | Physical activity level & consistency |
| `weekday_weekend_activity_ratio` | Actigraphy | Routine consistency across the week |
| `BMI` | Physical exam | Body composition |
| `BIA-BIA_Fat`, `BIA-BIA_FFMI`, `BIA-BIA_SMM` | Bioelectrical impedance analysis | Body composition (fat mass, fat-free mass index, skeletal muscle mass) |
| `PAQ_Total` | Physical Activity Questionnaire (self/parent-report) | Self-reported activity level, complementing the objective wearable measure |

**Held back for characterisation only (notebook 04):** `Basic_Demos-Age`, `Basic_Demos-Sex`, `PCIAT-PCIAT_Total`, `sii`, `CGAS-CGAS_Score`, `SDS-SDS_Total_T`, `PreInt_EduHx-computerinternet_hoursday`.

In [ ]:
df = train.merge(actigraphy_summary, on="id", how="left")

# harmonise PAQ_A / PAQ_C (age-gated instruments) into a single lifestyle feature
df["PAQ_Total"] = df["PAQ_A-PAQ_A_Total"].combine_first(df["PAQ_C-PAQ_C_Total"])

# prefer the direct physical-exam BMI, falling back to BIA-derived BMI
df["BMI"] = df["Physical-BMI"].combine_first(df["BIA-BIA_BMI"])

CLUSTER_FEATURES = [
    "sleep_duration_mean",
    "sleep_duration_std",
    "activity_intensity_mean",
    "activity_intensity_std",
    "weekday_weekend_activity_ratio",
    "BMI",
    "BIA-BIA_Fat",
    "BIA-BIA_FFMI",
    "BIA-BIA_SMM",
    "PAQ_Total",
]

CHARACTERISATION_VARS = [
    "Basic_Demos-Age",
    "Basic_Demos-Sex",
    "PCIAT-PCIAT_Total",
    "sii",
    "CGAS-CGAS_Score",
    "SDS-SDS_Total_T",
    "PreInt_EduHx-computerinternet_hoursday",
]

print("Missingness of clustering input features:")
display(df[CLUSTER_FEATURES].isna().mean().mul(100).round(1).rename("pct_missing").to_frame())


We require at least the actigraphy-derived features to be present (a child must have worn the device to contribute a lifestyle profile), then median-impute any remaining sporadic gaps in the body-composition / questionnaire columns. Median imputation is chosen over mean because several body-composition variables are right-skewed; it is applied only to the small residual missingness left after dropping actigraphy-less rows, and is documented here as a modelling choice rather than silently applied.

In [ ]:
has_actigraphy = df["sleep_duration_mean"].notna() & df["activity_intensity_mean"].notna()
print(f"Rows with usable actigraphy: {has_actigraphy.sum()} / {len(df)}")

model_df = df.loc[has_actigraphy, ["id"] + CLUSTER_FEATURES + CHARACTERISATION_VARS].reset_index(drop=True)

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(model_df[CLUSTER_FEATURES])
X_imputed = pd.DataFrame(X_imputed, columns=CLUSTER_FEATURES, index=model_df.index)

print(f"\nFinal modelling sample: {model_df.shape[0]} children, {len(CLUSTER_FEATURES)} lifestyle features")
display(X_imputed.describe().T)


## 5. Exploratory Data Analysis

Distributions and pairwise correlations of the engineered lifestyle features, before standardisation.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes.flat, CLUSTER_FEATURES):
    sns.histplot(X_imputed[col], kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col, fontsize=10)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 7))
corr = X_imputed.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Pairwise correlation of lifestyle features")
plt.tight_layout()
plt.show()


## Save artifacts for downstream notebooks

Everything the rest of the pipeline needs from this notebook: the imputed feature matrix, the full modelling table (including the held-back characterisation variables), and the two feature-name lists.

In [ ]:
artifact = {
    "model_df": model_df,
    "X_imputed": X_imputed,
    "CLUSTER_FEATURES": CLUSTER_FEATURES,
    "CHARACTERISATION_VARS": CHARACTERISATION_VARS,
}

with open(ARTIFACT_DIR / "01_data_preparation.pkl", "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved artifact -> {ARTIFACT_DIR / '01_data_preparation.pkl'}")
print(f"  model_df: {model_df.shape}, X_imputed: {X_imputed.shape}")
